In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "daily_model_v2.csv"
MODEL_DIR = ROOT / "models"
MODEL_PATH = MODEL_DIR / "mindwatch_v2.joblib"

TARGET = "stress"

FEATURES = [
    "avg_rmssd",
    "overall_score",
    "daily_steps",
    "avg_rmssd_dev"
]

RIDGE_ALPHA = 100

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
print("Participants:", df["participant_id"].nunique())
print("Target:", TARGET)
print("Features:", FEATURES)
print("Model output:", MODEL_PATH)

Dataset shape: (3118, 11)
Participants: 35
Target: stress
Features: ['avg_rmssd', 'overall_score', 'daily_steps', 'avg_rmssd_dev']
Model output: E:\schoolworks\y3s1\DIP\MindWatch-ML\models\mindwatch_v2.joblib


In [2]:
selection_columns = [
    "delta_hrv",
    "delta_sleep",
    "delta_steps",
    "delta_stress"
]

personal_df = (
    df.dropna(subset=selection_columns)
      .sort_values(["participant_id", "date"])
      .copy()
)

personal_datasets = {}

for pid, group in personal_df.groupby("participant_id"):

    #require at least 30 usable records
    if len(group) < 30:
        continue

    split_idx = int(len(group) * 0.8)

    personal_datasets[pid] = {
        "train": group.iloc[:split_idx].copy(),
        "test": group.iloc[split_idx:].copy()
    }

print("Eligible participants:", len(personal_datasets))

print(
    "Training samples:",
    sum(len(v["train"]) for v in personal_datasets.values())
)

print(
    "Testing samples:",
    sum(len(v["test"]) for v in personal_datasets.values())
)

Eligible participants: 14
Training samples: 693
Testing samples: 180


In [4]:
from sklearn.metrics import mean_absolute_error

personal_models = {}
personal_results = []

for pid, datasets in personal_datasets.items():

    train = datasets["train"].copy()
    test = datasets["test"].copy()

    history = pd.concat(
        [
            train.assign(_split="train"),
            test.assign(_split="test")
        ],
        ignore_index=True
    ).sort_values("date").reset_index(drop=True)

    historical_hrv_mean = (
        history["avg_rmssd"]
        .shift(1)
        .expanding(min_periods=7)
        .mean()
    )

    history["avg_rmssd_dev"] = (
        history["avg_rmssd"] - historical_hrv_mean
    )

    train = history[
        history["_split"] == "train"
    ].copy()

    test = history[
        history["_split"] == "test"
    ].copy()

    train = train.dropna(subset=[TARGET])
    test = test.dropna(subset=[TARGET])

    if len(train) < 15 or len(test) < 3:
        continue

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA)
    )

    model.fit(
        train[FEATURES],
        train[TARGET]
    )

    predictions = np.clip(
        model.predict(test[FEATURES]),
        0,
        100
    )

    baseline_predictions = np.full(
        len(test),
        train[TARGET].mean()
    )

    ridge_mae = mean_absolute_error(
        test[TARGET],
        predictions
    )

    baseline_mae = mean_absolute_error(
        test[TARGET],
        baseline_predictions
    )

    personal_models[pid] = model

    personal_results.append({
        "participant_id": pid,
        "train_samples": len(train),
        "test_samples": len(test),
        "ridge_mae": ridge_mae,
        "baseline_mae": baseline_mae,
        "improvement": baseline_mae - ridge_mae
    })

personal_results_df = pd.DataFrame(personal_results)

display(personal_results_df.round(3))

print("Trained models:", len(personal_models))

print(
    "Average Ridge MAE:",
    round(personal_results_df["ridge_mae"].mean(), 3)
)

print(
    "Average Baseline MAE:",
    round(personal_results_df["baseline_mae"].mean(), 3)
)

print(
    "Participants improved:",
    (personal_results_df["improvement"] > 0).sum(),
    "/",
    len(personal_results_df)
)

,participant_id,train_samples,test_samples,ridge_mae,baseline_mae,improvement
0,5,69,18,22.886,24.534,1.647
1,7,51,13,15.612,15.443,-0.169
2,8,62,16,19.705,18.637,-1.068
3,13,36,10,15.174,15.283,0.110
4,19,30,8,14.632,14.800,0.168
5,20,24,6,5.624,5.667,0.042
6,24,72,19,32.725,32.897,0.172
7,26,48,13,6.051,5.880,-0.171
8,27,62,16,24.108,25.619,1.511
9,28,45,12,11.539,12.326,0.787


Trained models: 14
Average Ridge MAE: 17.243
Average Baseline MAE: 17.547
Participants improved: 9 / 14


In [ ]:
#cold-start uses current-day physiological measurements only
#historical HRV deviation is excluded because new users
#may not have sufficient personal history

from sklearn.model_selection import GroupKFold

COLD_START_FEATURES = [
    "avg_rmssd",
    "overall_score",
    "daily_steps"
]

cold_start_df = (
    df.dropna(subset=[TARGET])
      .sort_values(["participant_id", "date"])
      .copy()
)

X_cold = cold_start_df[COLD_START_FEATURES]
y_cold = cold_start_df[TARGET]
groups = cold_start_df["participant_id"]

group_cv = GroupKFold(n_splits=5)

cold_start_splits = list(
    group_cv.split(X_cold, y_cold, groups)
)

print("Cold-start samples:", len(cold_start_df))
print("Participants:", groups.nunique())
print("Validation folds:", len(cold_start_splits))

for fold, (train_idx, test_idx) in enumerate(
    cold_start_splits, start=1
):
    train_participants = set(groups.iloc[train_idx])
    test_participants = set(groups.iloc[test_idx])

    print(
        f"Fold {fold}: "
        f"{len(train_participants)} train participants, "
        f"{len(test_participants)} test participants, "
        f"overlap={len(train_participants & test_participants)}"
    )

Cold-start samples: 3118
Participants: 35
Validation folds: 5
Fold 1: 28 train participants, 7 test participants, overlap=0
Fold 2: 28 train participants, 7 test participants, overlap=0
Fold 3: 28 train participants, 7 test participants, overlap=0
Fold 4: 28 train participants, 7 test participants, overlap=0
Fold 5: 28 train participants, 7 test participants, overlap=0


In [ ]:
from sklearn.metrics import mean_absolute_error

cold_start_results = []
cold_start_predictions = []

for fold, (train_idx, test_idx) in enumerate(
    cold_start_splits, start=1
):
    X_train = X_cold.iloc[train_idx]
    X_test = X_cold.iloc[test_idx]

    y_train = y_cold.iloc[train_idx]
    y_test = y_cold.iloc[test_idx]

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA)
    )

    model.fit(X_train, y_train)

    predictions = np.clip(
        model.predict(X_test),
        0,
        100
    )

    baseline = np.full(
        len(y_test),
        y_train.mean()
    )

    ridge_mae = mean_absolute_error(
        y_test, predictions
    )

    baseline_mae = mean_absolute_error(
        y_test, baseline
    )

    cold_start_results.append({
        "fold": fold,
        "ridge_mae": ridge_mae,
        "baseline_mae": baseline_mae,
        "improvement": baseline_mae - ridge_mae
    })

    cold_start_predictions.append(
        pd.DataFrame({
            "participant_id": groups.iloc[test_idx].values,
            "actual": y_test.values,
            "predicted": predictions
        })
    )

cold_start_results_df = pd.DataFrame(
    cold_start_results
)

cold_start_predictions_df = pd.concat(
    cold_start_predictions,
    ignore_index=True
)

display(cold_start_results_df.round(3))

print(
    "Average Cold-start Ridge MAE:",
    round(cold_start_results_df["ridge_mae"].mean(), 3)
)

print(
    "Average Population Baseline MAE:",
    round(cold_start_results_df["baseline_mae"].mean(), 3)
)

print(
    "Folds improved:",
    (cold_start_results_df["improvement"] > 0).sum(),
    "/ 5"
)

print(
    "Prediction standard deviation:",
    round(cold_start_predictions_df["predicted"].std(), 3)
)

print(
    "Actual standard deviation:",
    round(cold_start_predictions_df["actual"].std(), 3)
)

#MAE too high, thus cold-start is desserted

,fold,ridge_mae,baseline_mae,improvement
0,1,26.177,26.173,-0.004
1,2,17.853,17.592,-0.261
2,3,20.614,20.584,-0.031
3,4,19.522,18.559,-0.964
4,5,22.412,22.569,0.157


Average Cold-start Ridge MAE: 21.316
Average Population Baseline MAE: 21.095
Folds improved: 1 / 5
Prediction standard deviation: 2.978
Actual standard deviation: 25.65


In [7]:
MIN_CALIBRATION_SAMPLES = 30
RECOMMENDED_CALIBRATION_SAMPLES = 60

def check_calibration_status(user_data):

    required_columns = [
        "date",
        TARGET,
        *FEATURES[:3]
    ]

    missing_columns = [
        col for col in required_columns
        if col not in user_data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    data = user_data.copy()

    data["date"] = pd.to_datetime(
        data["date"], errors="coerce"
    )

    data = data.dropna(
        subset=["date", TARGET]
    )

    # Keep valid stress scores
    data = data[
        data[TARGET].between(0, 100)
    ]

    data = data.dropna(
        subset=FEATURES[:3],
        how="all"
    )

    data["date"] = data["date"].dt.normalize()

    data = (
        data.sort_values("date")
            .drop_duplicates(
                subset=["date"],
                keep="last"
            )
    )

    n_samples = len(data)

    if n_samples < MIN_CALIBRATION_SAMPLES:
        status = "collecting"
        can_train = False

    elif n_samples < RECOMMENDED_CALIBRATION_SAMPLES:
        status = "initial_ready"
        can_train = True

    else:
        status = "recommended_ready"
        can_train = True

    return {
        "status": status,
        "valid_samples": n_samples,
        "can_train": can_train,
        "samples_until_initial": max(
            0, MIN_CALIBRATION_SAMPLES - n_samples
        ),
        "samples_until_recommended": max(
            0, RECOMMENDED_CALIBRATION_SAMPLES - n_samples
        )
    }


example_pid = 5

example_user = (
    df[df["participant_id"] == example_pid]
    .sort_values("date")
    .head(45)
    .copy()
)

status = check_calibration_status(example_user)

print("Participant:", example_pid)
print("Calibration status:", status["status"])
print("Valid samples:", status["valid_samples"])
print("Can train:", status["can_train"])
print(
    "Samples until recommended:",
    status["samples_until_recommended"]
)

Participant: 5
Calibration status: initial_ready
Valid samples: 38
Can train: True
Samples until recommended: 22


In [9]:
def train_personalized_ridge(user_data):

    status = check_calibration_status(user_data)

    if not status["can_train"]:
        raise ValueError(
            "Insufficient calibration data. "
            f"Valid samples: {status['valid_samples']}"
        )

    data = user_data.copy()

    data["date"] = pd.to_datetime(data["date"])
    data["date"] = data["date"].dt.normalize()

    data = (
        data.sort_values("date")
            .drop_duplicates("date", keep="last")
            .reset_index(drop=True)
    )

    previous_hrv_mean = (
        data["avg_rmssd"]
        .shift(1)
        .expanding(min_periods=7)
        .mean()
    )

    data["avg_rmssd_dev"] = (
        data["avg_rmssd"] - previous_hrv_mean
    )

    data = data.dropna(subset=[TARGET])

    data = data[
        data[TARGET].between(0, 100)
    ]

    data = data.dropna(
        subset=FEATURES[:3],
        how="all"
    )

    X = data[FEATURES]
    y = data[TARGET]

    assert TARGET not in X.columns
    assert "delta_stress" not in X.columns
    assert "anxiety" not in X.columns

    missing_features = [
        col for col in FEATURES
        if X[col].notna().sum() == 0
    ]

    if missing_features:
        raise ValueError(
            f"Features without valid observations: {missing_features}"
        )

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=RIDGE_ALPHA)
    )

    model.fit(X, y)

    return {
        "model": model,
        "features": FEATURES.copy(),
        "training_samples": len(X),
        "calibration_status": status["status"]
    }

example_user = df[
    df["participant_id"] == 5
].sort_values("date").head(45).copy()

trained_user = train_personalized_ridge(example_user)

print("Personalized Ridge trained successfully")
print("Training samples:", trained_user["training_samples"])
print("Features:", trained_user["features"])
print("Calibration status:", trained_user["calibration_status"])

Personalized Ridge trained successfully
Training samples: 38
Features: ['avg_rmssd', 'overall_score', 'daily_steps', 'avg_rmssd_dev']
Calibration status: initial_ready


In [ ]:
def predict_personalized_stress(
    trained_user,
    avg_rmssd,
    overall_score,
    daily_steps,
    hrv_history
):
    """
    Predict current-day stress using physiological data only.

    hrv_history:
        Previous daily HRV values, ordered chronologically.
        Must exclude the current day's HRV.
    """

    model = trained_user["model"]

    history = pd.to_numeric(
        pd.Series(hrv_history),
        errors="coerce"
    )

    if history.notna().sum() >= 7 and pd.notna(avg_rmssd):
        historical_mean = history.mean()
        hrv_deviation = float(avg_rmssd) - historical_mean
    else:
        hrv_deviation = np.nan

    X_new = pd.DataFrame([{
        "avg_rmssd": avg_rmssd,
        "overall_score": overall_score,
        "daily_steps": daily_steps,
        "avg_rmssd_dev": hrv_deviation
    }])

    X_new = X_new[trained_user["features"]]

    prediction = model.predict(X_new)[0]

    prediction = float(np.clip(prediction, 0, 100))

    return prediction

example_history = (
    example_user["avg_rmssd"]
    .iloc[:-1]
    .tolist()
)

latest_record = example_user.iloc[-1]

predicted_stress = predict_personalized_stress(
    trained_user=trained_user,
    avg_rmssd=latest_record["avg_rmssd"],
    overall_score=latest_record["overall_score"],
    daily_steps=latest_record["daily_steps"],
    hrv_history=example_history
)

print("Predicted stress:", round(predicted_stress, 2))
print("Prediction range: 0-100")

Predicted stress: 7.04
Prediction range: 0-100


In [12]:
actual_stress = float(latest_record[TARGET])

print("Predicted stress:", round(predicted_stress, 2))
print("Actual stress:", round(actual_stress, 2))
print(
    "Absolute error:",
    round(abs(predicted_stress - actual_stress), 2)
)

print(
    "Personal training mean:",
    round(example_user[TARGET].mean(), 2)
)

Predicted stress: 7.04
Actual stress: 7.0
Absolute error: 0.04
Personal training mean: 6.53


In [14]:
from datetime import datetime, timezone

deployment_package = {
    "model_name": "MindWatch V2 Personalized Ridge",
    "version": "2.0",

    "target": "stress",
    "stress_range": (0, 100),

    "features": FEATURES.copy(),

    "model_type": "Ridge",
    "ridge_alpha": RIDGE_ALPHA,
    "imputer_strategy": "median",
    "scaler": "StandardScaler",

    "min_calibration_samples": 30,
    "recommended_calibration_samples": 60,

    "hrv_deviation_min_history": 7,

    "personalized_training": True,
    "requires_personal_calibration": True,
    "cold_start_supported": False,

    "evaluation": {
        "personalized_ridge_mae": 17.243,
        "personal_mean_baseline_mae": 17.547,
        "participants_evaluated": 14,
        "participants_improved": 9
    },

    "created_at": datetime.now(timezone.utc).isoformat()
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    deployment_package,
    MODEL_PATH
)

loaded_package = joblib.load(MODEL_PATH)

print("Export successful!")
print("Model path:", MODEL_PATH)
print("Model name:", loaded_package["model_name"])
print("Features:", loaded_package["features"])
print(
    "Minimum calibration samples:",
    loaded_package["min_calibration_samples"]
)
print(
    "Recommended calibration samples:",
    loaded_package["recommended_calibration_samples"]
)

Export successful!
Model path: E:\schoolworks\y3s1\DIP\MindWatch-ML\models\mindwatch_v2.joblib
Model name: MindWatch V2 Personalized Ridge
Features: ['avg_rmssd', 'overall_score', 'daily_steps', 'avg_rmssd_dev']
Minimum calibration samples: 30
Recommended calibration samples: 60


In [17]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_template = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=100))
])

deployment_package = {
    "name": "MindWatch V2 Personalized Ridge",
    "version": "2.0",
    "task": "same_day_stress_estimation",

    "features": [
        "avg_rmssd",
        "overall_score",
        "daily_steps",
        "avg_rmssd_dev"
    ],

    "target": "stress",
    "output_range": (0, 100),

    "model_template": ridge_template,

    "calibration": {
        "minimum_samples": 30,
        "recommended_samples": 60,
        "hrv_history_minimum": 7
    },

    "evaluation": {
        "personalized_mae": 17.243,
        "baseline_mae": 17.547,
        "participants": 14,
        "participants_improved": 9
    }
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(deployment_package, MODEL_PATH)

loaded = joblib.load(MODEL_PATH)

print("Export successful:", MODEL_PATH)
print("Task:", loaded["task"])
print("Features:", loaded["features"])
print("Model:", loaded["model_template"])
print("Minimum samples:", loaded["calibration"]["minimum_samples"])


Export successful: E:\schoolworks\y3s1\DIP\MindWatch-ML\models\mindwatch_v2.joblib
Task: same_day_stress_estimation
Features: ['avg_rmssd', 'overall_score', 'daily_steps', 'avg_rmssd_dev']
Model: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()), ('ridge', Ridge(alpha=100))])
Minimum samples: 30
